# BUG: `pivot_table` performance with MultiIndex 5x worse than it should be.

In [1]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
import pandas as pd

In [3]:
from itertools import product
from string import ascii_lowercase

import numpy as np
from pandas import DataFrame, Index, MultiIndex, date_range

level0 = ["foo", "bar", "baz"]
level1 = list(range(2**6))
level2 = date_range("1990", "2022")

index = MultiIndex.from_product(
    [level0, level1, level2], names=["outer", "inner", "time"]
)

variables = list(ascii_lowercase)
df = DataFrame(np.random.randn(len(index), 1), index=index, columns=["value"])
df["variable"] = np.random.choice(variables, len(index))
df

value variable
outer inner time                         
foo   0     1990-01-01 -0.877869        t
            1990-01-02  1.056480        u
            1990-01-03 -0.389669        o
            1990-01-04 -2.147823        w
            1990-01-05 -0.589675        y
...                          ...      ...
baz   63    2021-12-28  0.443157        k
            2021-12-29  1.001620        r
            2021-12-30  0.443647        r
            2021-12-31  0.199754        g
            2022-01-01 -1.398097        u

[2244288 rows x 2 columns]

In [7]:
7, 168, 12685

(7, 168, 12685)

In [10]:
df.pivot_table(index=df.index.names, columns="variable", values="value", dropna=False)

variable                 a         b   c   d   e   f         g         h  \
outer inner time                                                           
bar   0     1990-01-01 NaN       NaN NaN NaN NaN NaN       NaN       NaN   
            1990-01-02 NaN       NaN NaN NaN NaN NaN       NaN       NaN   
            1990-01-03 NaN       NaN NaN NaN NaN NaN       NaN       NaN   
            1990-01-04 NaN       NaN NaN NaN NaN NaN       NaN       NaN   
            1990-01-05 NaN       NaN NaN NaN NaN NaN       NaN       NaN   
...                     ..       ...  ..  ..  ..  ..       ...       ...   
foo   63    2021-12-28 NaN       NaN NaN NaN NaN NaN       NaN       NaN   
            2021-12-29 NaN  0.172667 NaN NaN NaN NaN       NaN       NaN   
            2021-12-30 NaN       NaN NaN NaN NaN NaN       NaN -0.954278   
            2021-12-31 NaN       NaN NaN NaN NaN NaN  0.757283       NaN   
            2022-01-01 NaN       NaN NaN NaN NaN NaN       NaN       NaN   

variable                       i   j  ...   q   r   s         t         u   v  \
outer inner time                      ...                                       
bar   0     1990-01-01  0.281905 NaN  ... NaN NaN NaN       NaN       NaN NaN   
            1990-01-02       NaN NaN  ... NaN NaN NaN -1.126152       NaN NaN   
            1990-01-03 -0.181881 NaN  ... NaN NaN NaN       NaN       NaN NaN   
            1990-01-04       NaN NaN  ... NaN NaN NaN       NaN       NaN NaN   
            1990-01-05       NaN NaN  ... NaN NaN NaN       NaN       NaN NaN   
...                          ...  ..  ...  ..  ..  ..       ...       ...  ..   
foo   63    2021-12-28       NaN NaN  ... NaN NaN NaN       NaN       NaN NaN   
            2021-12-29       NaN NaN  ... NaN NaN NaN       NaN       NaN NaN   
            2021-12-30       NaN NaN  ... NaN NaN NaN       NaN       NaN NaN   
            2021-12-31       NaN NaN  ... NaN NaN NaN       NaN       NaN NaN   
            2022-01-01       NaN NaN  ... NaN NaN NaN       NaN  2.640404 NaN   

variable                 w   x   y        z  
outer inner time                             
bar   0     1990-01-01 NaN NaN NaN      NaN  
            1990-01-02 NaN NaN NaN      NaN  
            1990-01-03 NaN NaN NaN      NaN  
            1990-01-04 NaN NaN NaN  1.05992  
            1990-01-05 NaN NaN NaN      NaN  
...                     ..  ..  ..      ...  
foo   63    2021-12-28 NaN NaN NaN      NaN  
            2021-12-29 NaN NaN NaN      NaN  
            2021-12-30 NaN NaN NaN      NaN  
            2021-12-31 NaN NaN NaN      NaN  
            2022-01-01 NaN NaN NaN      NaN  

[2244288 rows x 26 columns]

In [8]:
%%timeit -n 1 -r 1
df.pivot_table(index=df.index.names, columns="variable", values="value", dropna=False)

1.71 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [9]:
%%timeit -n 1 -r 1
x = df.pivot_table(index=df.index, columns="variable", values="value", dropna=False)
x.index = MultiIndex.from_tuples(x.index, names=df.index.names)

9.02 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [33]:
%%timeit -n 1 -r 1
df.pivot_table(index=df.index.names, columns="variable", values="value")

1.68 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [18]:
ascii_letters

'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ'

In [4]:
df = pd.DataFrame(
    {
        "number": ["one", "two", "one", "three"],
        "letter": ["a", "a", "b", "b"],
        "y": [1, 2, np.nan, np.nan],
        "x": [1, 2, 3, 4],
    }
)

,number,letter,y,x
0,one,a,1.0,1
1,two,a,2.0,2
2,one,b,NaN,3
3,three,b,NaN,4


In [6]:
result = df.pivot_table(
    values="y", columns=["number", "letter"], index="x", observed=True, dropna=False
)

number  one     three      two    
letter    a   b     a   b    a   b
x                                 
1       1.0 NaN   NaN NaN  NaN NaN
2       NaN NaN   NaN NaN  2.0 NaN
3       NaN NaN   NaN NaN  NaN NaN
4       NaN NaN   NaN NaN  NaN NaN